<div style="
    width:100%;
    max-width:1080px;
    margin:24px 0 32px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #c2410c 0%, #f59e0b 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h1 style="margin:0 0 14px 0; font-size:34px; line-height:1.2; font-weight:700; color:#ffffff;">El Loop de Validación</h1>
<p style="margin:0; color:#cbd5e1; font-size:18px;">Entrenar sin validar deja ciega una parte crucial del proceso: saber si el modelo mejora realmente fuera de los datos con los que ajusta parámetros. Este notebook introduce el loop de validación como contraste necesario.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Por qué importa</div>
    <div style="color:#f8fafc;">La validación ayuda a detectar sobreajuste, estancamiento o mejoras aparentes que no generalizan.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Qué vas a aprender aquí</div>
    <div style="color:#f8fafc;">A distinguir el loop de validación del loop de entrenamiento y a entender por qué no cumplen la misma función.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Regla práctica que debes recordar</div>
    <div style="color:#f8fafc;">Una pérdida de entrenamiento mejor no implica automáticamente un modelo mejor sobre datos no vistos.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Contexto mínimo</div>
    <div style="color:#f8fafc;">Validar significa evaluar periódicamente el modelo en un conjunto separado sin actualizar sus parámetros.</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:36px 0 42px 0;">

In [ ]:
# Bloque 1 — Tensores de entrada y etiquetas de validación
"""Este bloque crea tensores que representan las entradas y las etiquetas del conjunto de validación.
Este bloque importa "torch" para trabajar con tensores en PyTorch.
La variable "entradas_val" contiene muestras bidimensionales en forma de tensor que sirven como datos de entrada.

Aplicamos el método ".unsqueeze()" a las etiquetas para que tengan una dimensión adicional, convirtiéndolas en tensores columna.
El argumento "1" indica que se debe agregar una dimensión adicional en la posición 1 del tensor.
La variable "etiquetas_val" contiene valores asociados a cada muestra en forma de tensor con una dimensión adicional en 
la posición 1 y define el objetivo de la evaluación.
Estos tensores permiten demostrar cómo se evalúa el modelo sin actualizar parámetros."""

import torch

entradas_val = torch.tensor([[2.0, 1.0],
                             [4.0, 3.0],
                             [6.0, 5.0]])

etiquetas_val = torch.tensor([1.0, 0.0, 1.0]).unsqueeze(1)

entradas_val, etiquetas_val # Muestra los tensores de entrada y etiquetas de validación para su verificación.


In [ ]:
# Bloque 2 — Definición de una red neuronal
"""Este bloque define una red neuronal artificial utilizando nn.Module.
Este bloque importa "torch.nn" para definir módulos y capas de redes neuronales.

Definimos una clase personalizada llamada "RedValidacion()" que representa un modelo de red neuronal simple.
La clase "RedValidacion()" hereda de "nn.Module", que es la clase base utilizada para construir redes neuronales en PyTorch.

Definimos la función "def __init__()", que inicializa la estructura interna del modelo.
El parámetro "self" de la función hace referencia a la instancia actual de la clase "RedValidacion()".

Utilizamos "super().__init__()" para invocar el constructor de la clase base "nn.Module".     
Esto asegura que la clase base se inicialice correctamente cada vez que se crea una instancia del modelo.

Aplicamos la clase "nn.Linear()" a la variable de instancia "self.capa1" para definir la primera capa lineal.
Los argumentos "2, 4" de "nn.Linear()" indican que la capa lineal recibe 2 entradas y produce 4 salidas.
Aplicamos la clase "nn.Linear()" a la variable de instancia "self.capa2" para definir la segunda capa lineal.
Los argumentos "4, 1" de "nn.Linear()" indican que la capa lineal recibe 4 entradas y produce 1 salida.

Definimos la función "def forward()" que especifica cómo fluye la entrada a través de las capas del modelo.
El método ".forward()" especifica cómo se propaga la información a través de la red.
Los parámetros "self" y "x" representan la instancia del modelo y los datos de entrada, respectivamente.

Asignamos el resultado de llamar a "self.capa1()" a la variable "x", aplicando la primera capa lineal a la entrada.
Aplicamos la función de activación "torch.relu()" a la variable "x" para introducir no linealidad en la red.
Asignamos el resultado de llamar a "self.capa2()" a la variable "x", aplicando la segunda capa lineal a la salida de la activación.
Utilizamos la instrucción "return" para devolver la salida del modelo aplicando las capas lineales a la entrada.

En todos los casos se pasa "x" como parámetro a las funciones correspondientes para que la información fluya a través de la red.
Además, la variable "x" se actualiza con la salida de cada operación, asegurando que la información fluya correctamente a través de la red.
La variable "red" almacena una instancia de la clase "RedValidacion()", representando el modelo de red neuronal definido.
La estructura permite demostrar cómo la red genera predicciones sin modificar sus parámetros."""

import torch.nn as nn

class RedValidacion(nn.Module):
    def __init__(self):
        super().__init__()
        self.capa1 = nn.Linear(2, 4)
        self.capa2 = nn.Linear(4, 1)

    def forward(self, x):
        x = self.capa1(x)
        x = torch.relu(x)
        x = self.capa2(x)
        return x

red = RedValidacion()

red # Muestra la estructura de la red neuronal definida.


In [ ]:
# Bloque 3 — Definición de la función de pérdida
"""Este bloque define la función de pérdida utilizada en el loop de validación.
Aplicamos la clase "nn.BCEWithLogitsLoss()" a la variable "criterio" para crear un objeto calculador de función de pérdida.
La variable "criterio" almacena el objeto que evalúa el rendimiento del modelo."""

criterio = nn.BCEWithLogitsLoss()

criterio # Muestra la estructura de la función de pérdida definida.


In [ ]:
# Bloque 4 — Loop de validación
"""Este bloque realiza una evaluación de validación sobre todos los datos a la vez.
La variable "perdidas_val" se inicializa como una lista vacía "[]" para almacenar los valores de pérdida obtenidos durante el proceso de validación.

Utilizamos la palabra clave "with" para crear un contexto en el que el cálculo de gradientes está desactivado.
Aplicamos la clase "torch.no_grad()" para desactivar el cálculo de gradientes durante la validación en este contexto.
El contexto "with torch.no_grad():" desactiva el cálculo de gradientes para evitar el almacenamiento innecesario de información para backpropagation.

En esta validación se calculan los logits del modelo para el conjunto de datos de validación y se calcula la pérdida correspondiente 
al conjunto de datos de validación.
Además, se almacena el valor de la pérdida en la lista "perdidas_val" para su posterior análisis.
En este ejemplo la red se acaba de crear y no se ha entrenado. El contexto "no_grad()" no cambia el modelo a modo de evaluación."""

perdidas_val = []

with torch.no_grad():
    predicciones_val = red(entradas_val)
    perdida_val = criterio(predicciones_val, etiquetas_val) 
    perdidas_val.append(float(perdida_val.item())) # La función "item()" obtiene el valor escalar del tensor de pérdida.

perdidas_val # Muestra los valores de pérdida obtenidos durante la validación.


In [ ]:
# Bloque 5 — Resultados del loop de validación
"""Este bloque organiza los elementos principales del loop de validación.
La variable "resultado" almacena en forma de diccionario las pérdidas registradas y las predicciones generadas por el modelo.
La estructura permite visualizar el resultado de evaluar la red recién creada sin actualizar sus parámetros."""

resultado = {
    "perdidas_validacion": perdidas_val,
    "predicciones_validacion": predicciones_val.tolist()
}

resultado # Muestra el diccionario que contiene las pérdidas y predicciones de validación.


<div style="
    width:100%;
    max-width:1080px;
    margin:40px 0 24px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #0f766e 0%, #2dd4bf 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h2 style="margin:0 0 14px 0; font-size:30px; line-height:1.2; font-weight:700; color:#ffffff;">Conclusiones</h2>
<p style="margin:0; color:#cbd5e1; font-size:18px;">En este notebook has visto que validar no es repetir el entrenamiento, sino observar el modelo desde fuera del ajuste directo.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Qué conviene recordar</div>
    <div style="color:#f8fafc;">La validación permite juzgar si el aprendizaje se traduce en mejor generalización.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Error frecuente a evitar</div>
    <div style="color:#f8fafc;">Confundir señales de mejora en entrenamiento con evidencia suficiente de buen rendimiento global.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente paso natural</div>
    <div style="color:#f8fafc;">Detenerse en las Métricas Básicas para dar continuidad directa al hilo del temario.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Conexión con el siguiente concepto</div>
    <div style="color:#f8fafc;">El siguiente notebook se centra en las Métricas Básicas y amplía de forma natural lo que acabas de trabajar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente notebook del temario</div>
    <div style="color:#f8fafc;">05_Entrenamiento_de_redes/04_Las_metricas_basicas.ipynb</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:42px 0 24px 0;">